# Chapter 5: Well Performance and Artificial Lift

This notebook covers well performance analysis using NeqSim's `PipeBeggsAndBrills` model:
- Vertical Lift Performance (VLP) curves at different tubing sizes
- Operating point determination (IPR vs VLP intersection)
- Gas lift optimization concepts
- Effect of water cut on wellhead pressure

NeqSim provides rigorous multiphase pipe flow calculations using the Beggs & Brill correlation.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    NEQSIM_MODE = "pip"

# Always ensure jneqsim is available (works in both modes)
try:
    import neqsim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])

from neqsim import jneqsim
print(f"NeqSim ready (mode: {NEQSIM_MODE})")

# Common class shortcuts for convenience
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
SystemPrEos = jneqsim.thermo.system.SystemPrEos
SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

# Process equipment
Stream = jneqsim.process.equipment.stream.Stream
Separator = jneqsim.process.equipment.separator.Separator
ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Heater = jneqsim.process.equipment.heatexchanger.Heater
HeatExchanger = jneqsim.process.equipment.heatexchanger.HeatExchanger
Mixer = jneqsim.process.equipment.mixer.Mixer
Splitter = jneqsim.process.equipment.splitter.Splitter
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
Pump = jneqsim.process.equipment.pump.Pump
Expander = jneqsim.process.equipment.expander.Expander
Recycle = jneqsim.process.equipment.util.Recycle
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim2
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim2\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim2\src\main\resources
  3. C:\Users\ESOL\Documents\GitHub\neqsim2\target\neqsim-3.7.0.jar



JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim loaded via devtools (local dev mode)
NeqSim ready (mode: devtools)


In [2]:
import numpy as np
import matplotlib.pyplot as plt

# NeqSim class imports
Stream = jneqsim.process.equipment.stream.Stream
PipeBeggsAndBrills = jneqsim.process.equipment.pipeline.PipeBeggsAndBrills
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

plt.rcParams.update({'font.size': 12, 'figure.figsize': (10, 7)})

## Create Production Fluid

A typical oil well production fluid with gas, oil, and water components.

In [3]:
def create_well_fluid(water_fraction=0.0):
    """Create production fluid with optional water cut."""
    T_bh_K = 273.15 + 80.0  # Bottomhole temperature
    P_bh_bara = 200.0  # Bottomhole pressure

    fluid = SystemSrkEos(T_bh_K, P_bh_bara)
    fluid.addComponent("nitrogen", 0.3)
    fluid.addComponent("CO2", 1.0)
    fluid.addComponent("methane", 60.0)
    fluid.addComponent("ethane", 8.0)
    fluid.addComponent("propane", 5.0)
    fluid.addComponent("i-butane", 1.5)
    fluid.addComponent("n-butane", 2.5)
    fluid.addComponent("i-pentane", 1.0)
    fluid.addComponent("n-pentane", 1.0)
    fluid.addComponent("n-hexane", 3.0)
    fluid.addComponent("n-heptane", 5.0)
    fluid.addComponent("n-octane", 4.0)
    fluid.addComponent("n-nonane", 2.0)
    if water_fraction > 0.0:
        # Add water as a fraction of total moles
        hc_total = 94.3  # sum of HC components above
        water_moles = hc_total * water_fraction / (1.0 - water_fraction)
        fluid.addComponent("water", water_moles)
    fluid.setMixingRule("classic")
    fluid.setMultiPhaseCheck(True)
    return fluid

# Test the base fluid
test_fluid = create_well_fluid()
ops = ThermodynamicOperations(test_fluid)
ops.TPflash()
test_fluid.initProperties()
print(f"Phases: {test_fluid.getNumberOfPhases()}")
print(f"Gas density: {test_fluid.getPhase('gas').getDensity('kg/m3'):.2f} kg/m3")

Phases: 2
Gas density: 239.14 kg/m3


## Figure 1: Vertical Lift Performance (VLP) Curves

VLP curves show the relationship between bottomhole flowing pressure and production rate
for different tubing diameters. Larger tubing reduces friction but may cause velocity issues
(heading, liquid loading).

In [4]:
# Wellbore parameters
well_depth = 2500.0  # m
roughness = 2.5e-5   # m
diameters = [0.076, 0.10, 0.127]  # 3", 4", 5" tubing IDs in meters
diameter_labels = ['3.0 inch', '4.0 inch', '5.0 inch']
colors = ['blue', 'green', 'red']

# Flow rates to evaluate (kg/hr)
flow_rates = np.linspace(5000, 80000, 12)

fig, ax = plt.subplots(figsize=(10, 7))

for i, (diam, label, color) in enumerate(zip(diameters, diameter_labels, colors)):
    pwf_values = []
    valid_rates = []

    for rate in flow_rates:
        try:
            fluid = create_well_fluid()
            feed = Stream("well feed", fluid)
            feed.setFlowRate(float(rate), "kg/hr")
            feed.setTemperature(273.15 + 80.0, "K")
            feed.setPressure(200.0, "bara")

            pipe = PipeBeggsAndBrills("wellbore", feed)
            pipe.setPipeWallRoughness(roughness)
            pipe.setLength(well_depth)
            pipe.setElevation(-well_depth)  # negative = upward flow
            pipe.setDiameter(diam)

            process = ProcessSystem()
            process.add(feed)
            process.add(pipe)
            process.run()

            p_wh = pipe.getOutletStream().getPressure("bara")
            if p_wh > 5.0:  # Only valid if wellhead pressure is reasonable
                pwf_values.append(p_wh)
                valid_rates.append(rate / 1000.0)  # Convert to tonnes/hr
        except Exception as e:
            pass  # Skip failed calculations

    if valid_rates:
        ax.plot(valid_rates, pwf_values, '-o', color=color, linewidth=2, markersize=5,
                label=f'Tubing ID = {label}')

ax.set_xlabel('Flow Rate (tonnes/hr)', fontsize=14)
ax.set_ylabel('Wellhead Pressure (bara)', fontsize=14)
ax.set_title('Vertical Lift Performance (VLP) Curves - Different Tubing Sizes', fontsize=15)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch05_fig01_vlp_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 1 saved.")

Figure 1 saved.


C:\Users\ESOL\AppData\Local\Temp\ipykernel_40044\3004212568.py:54: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 2: Operating Point (IPR vs VLP Intersection)

The well operating point is where the IPR and VLP curves intersect. The IPR describes
the reservoir's ability to deliver fluid, while the VLP describes the tubing's ability
to lift it.

In [5]:
# IPR parameters
P_res = 250.0  # Reservoir pressure (bara)
PI = 0.8  # Productivity index (tonnes/hr/bar)

# IPR curve: Pwf vs flow rate
q_range = np.linspace(1, 150, 200)  # tonnes/hr
Pwf_ipr = P_res - q_range / PI  # linear PI model

# VLP curve from pipe simulation (4" tubing)
flow_rates_vlp = np.linspace(5000, 120000, 16)  # kg/hr
pwf_vlp = []
valid_rates_vlp = []

for rate in flow_rates_vlp:
    try:
        fluid = create_well_fluid()
        feed = Stream("well feed", fluid)
        feed.setFlowRate(float(rate), "kg/hr")
        feed.setTemperature(273.15 + 80.0, "K")
        feed.setPressure(float(P_res), "bara")

        pipe = PipeBeggsAndBrills("wellbore", feed)
        pipe.setPipeWallRoughness(2.5e-5)
        pipe.setLength(2500.0)
        pipe.setElevation(-2500.0)
        pipe.setDiameter(0.10)

        process = ProcessSystem()
        process.add(feed)
        process.add(pipe)
        process.run()

        # VLP: required bottomhole pressure = Pres - (Pres - outlet) is wrong
        # VLP gives wellhead pressure for a given BHP.
        # For intersection: we need BHP required to deliver rate through tubing
        # BHP_vlp = Pwh + dP_gravity - dP_friction (approx)
        # We use inlet - outlet pressure drop approach:
        p_in = feed.getPressure("bara")
        p_out = pipe.getOutletStream().getPressure("bara")
        dp_tubing = p_in - p_out  # Total pressure change in tubing

        # Required BHP to get 50 bara at wellhead
        P_wh_target = 50.0  # Minimum wellhead pressure
        Pwf_required = P_wh_target + dp_tubing

        pwf_vlp.append(Pwf_required)
        valid_rates_vlp.append(rate / 1000.0)  # tonnes/hr
    except Exception:
        pass

fig, ax = plt.subplots(figsize=(10, 7))

# IPR curve
mask_ipr = Pwf_ipr > 0
ax.plot(q_range[mask_ipr], Pwf_ipr[mask_ipr], 'b-', linewidth=2.5, label='IPR (PI model)')

# VLP curve
if valid_rates_vlp:
    ax.plot(valid_rates_vlp, pwf_vlp, 'r-s', linewidth=2.5, markersize=6,
            label=f'VLP (4" tubing, Pwh={P_wh_target:.0f} bara)')

    # Find approximate intersection
    vlp_interp = np.interp(q_range[mask_ipr], valid_rates_vlp, pwf_vlp)
    diff = np.abs(Pwf_ipr[mask_ipr] - vlp_interp)
    idx_cross = np.argmin(diff)
    q_op = q_range[mask_ipr][idx_cross]
    p_op = Pwf_ipr[mask_ipr][idx_cross]

    ax.plot(q_op, p_op, 'k*', markersize=20, zorder=5, label=f'Operating Point')
    ax.annotate(f'q = {q_op:.1f} t/hr\nPwf = {p_op:.0f} bara',
                xy=(q_op, p_op), xytext=(q_op + 15, p_op + 10),
                fontsize=11, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

ax.set_xlabel('Production Rate (tonnes/hr)', fontsize=14)
ax.set_ylabel('Bottomhole Flowing Pressure (bara)', fontsize=14)
ax.set_title('Operating Point: IPR vs VLP Intersection', fontsize=15)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, P_res * 1.1)
plt.tight_layout()
plt.savefig('../figures/ch05_fig02_operating_point.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 2 saved.")

Figure 2 saved.


C:\Users\ESOL\AppData\Local\Temp\ipykernel_40044\3626019648.py:83: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 3: Gas Lift Optimization

Gas lift reduces the hydrostatic head in the tubing by injecting gas, but excessive injection
increases friction losses. There is an optimal injection rate that maximizes oil production.

In [6]:
# Conceptual gas lift performance curve
# Based on typical behavior: production increases then plateaus/decreases
gas_injection_rate = np.linspace(0, 5.0, 100)  # MMscf/d

# Production response model: q = q_natural + gain * (1 - exp(-k*Qgi)) - friction_loss * Qgi^2
q_natural = 2000.0  # Natural flow rate (bbl/d)
gain = 4000.0       # Maximum additional production
k = 1.2             # Response factor
friction_coeff = 120.0  # Friction penalty coefficient

q_oil = q_natural + gain * (1.0 - np.exp(-k * gas_injection_rate)) - friction_coeff * gas_injection_rate**2

# Find optimum
idx_opt = np.argmax(q_oil)
q_opt_inj = gas_injection_rate[idx_opt]
q_opt_oil = q_oil[idx_opt]

fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(gas_injection_rate, q_oil, 'g-', linewidth=2.5, label='Oil Production Rate')
ax.axhline(y=q_natural, color='gray', linestyle='--', alpha=0.6, label=f'Natural flow = {q_natural:.0f} bbl/d')

# Mark optimum
ax.plot(q_opt_inj, q_opt_oil, 'r*', markersize=20, zorder=5, label='Optimal Injection Rate')
ax.axvline(x=q_opt_inj, color='red', linestyle=':', alpha=0.4)
ax.annotate(f'Optimum\nQgi = {q_opt_inj:.2f} MMscf/d\nq_oil = {q_opt_oil:.0f} bbl/d',
            xy=(q_opt_inj, q_opt_oil), xytext=(q_opt_inj + 0.8, q_opt_oil - 400),
            fontsize=11, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

# Shade regions
ax.fill_between(gas_injection_rate[:idx_opt+1], q_natural, q_oil[:idx_opt+1],
                alpha=0.1, color='green', label='Net gain region')
ax.fill_between(gas_injection_rate[idx_opt:], q_oil[idx_opt:], q_oil[idx_opt],
                alpha=0.1, color='red', label='Diminishing returns')

ax.set_xlabel('Gas Injection Rate (MMscf/d)', fontsize=14)
ax.set_ylabel('Oil Production Rate (bbl/d)', fontsize=14)
ax.set_title('Gas Lift Optimization - Injection Rate vs Production', fontsize=15)
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 5.0)
plt.tight_layout()
plt.savefig('../figures/ch05_fig03_gas_lift_optimization.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 3 saved.")

Figure 3 saved.


C:\Users\ESOL\AppData\Local\Temp\ipykernel_40044\1199891922.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 4: Wellhead Pressure vs Flow Rate at Different Water Cuts

Increasing water cut raises the hydrostatic head in the tubing (water is denser than oil/gas),
reducing the wellhead pressure for a given bottomhole pressure. This directly impacts
well deliverability.

In [7]:
# Water cut scenarios
water_cuts = [0.0, 0.10, 0.30, 0.50]
wc_labels = ['0% WC', '10% WC', '30% WC', '50% WC']
wc_colors = ['blue', 'green', 'orange', 'red']
flow_rates_wc = np.linspace(5000, 80000, 10)  # kg/hr

fig, ax = plt.subplots(figsize=(10, 7))

for wc, label, color in zip(water_cuts, wc_labels, wc_colors):
    pwh_values = []
    valid_rates = []

    for rate in flow_rates_wc:
        try:
            fluid = create_well_fluid(water_fraction=wc)
            feed = Stream("well feed", fluid)
            feed.setFlowRate(float(rate), "kg/hr")
            feed.setTemperature(273.15 + 80.0, "K")
            feed.setPressure(200.0, "bara")

            pipe = PipeBeggsAndBrills("wellbore", feed)
            pipe.setPipeWallRoughness(2.5e-5)
            pipe.setLength(2500.0)
            pipe.setElevation(-2500.0)
            pipe.setDiameter(0.10)  # 4 inch

            process = ProcessSystem()
            process.add(feed)
            process.add(pipe)
            process.run()

            p_wh = pipe.getOutletStream().getPressure("bara")
            if p_wh > 1.0:
                pwh_values.append(p_wh)
                valid_rates.append(rate / 1000.0)
        except Exception:
            pass

    if valid_rates:
        ax.plot(valid_rates, pwh_values, f'-o', color=color, linewidth=2,
                markersize=5, label=label)

ax.set_xlabel('Flow Rate (tonnes/hr)', fontsize=14)
ax.set_ylabel('Wellhead Pressure (bara)', fontsize=14)
ax.set_title('Wellhead Pressure vs Flow Rate at Different Water Cuts', fontsize=15)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch05_fig04_watercut_effect.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 4 saved.")

Figure 4 saved.


C:\Users\ESOL\AppData\Local\Temp\ipykernel_40044\2777976503.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

This chapter demonstrated key well performance concepts:

1. **VLP Curves**: Larger tubing diameters reduce friction losses but too-large tubing can lead to liquid loading. NeqSim's Beggs & Brill model captures the multiphase flow physics accurately.

2. **Operating Point**: The intersection of IPR and VLP defines the natural operating point. Changes in reservoir pressure, tubing size, or wellhead pressure shift this point.

3. **Gas Lift**: There is an optimal gas injection rate that maximizes production. Beyond this point, additional gas increases friction more than it reduces hydrostatic head.

4. **Water Cut Effect**: Increasing water production dramatically reduces wellhead pressure due to the higher density of water. This is a critical consideration for late-life field management.

### Key Takeaway
NeqSim's `PipeBeggsAndBrills` model provides rigorous multiphase flow calculations essential
for well performance analysis. Combined with thermodynamic fluid models, it enables accurate
prediction of well deliverability under various operating scenarios.